# Projeto CardioIA — Fase 2: Classificador Básico de Texto
## Triagem Clínica Automatizada com Machine Learning

**Objetivo:** Desenvolver um classificador que analisa frases de sintomas relatados por pacientes e classifica o nível de risco como **"alto risco"** ou **"baixo risco"**, simulando sistemas automatizados de triagem clínica.

**Pipeline do projeto:**
1. Leitura e exploração do dataset rotulado
2. Pré-processamento textual
3. Vetorização com TF-IDF (unigramas e bigramas)
4. Treinamento com Logistic Regression e Decision Tree
5. Avaliação com acurácia, precision, recall, F1-score e validação cruzada
6. Teste com frases inéditas
7. Análise de vieses e limitações

### 1. Importação das Bibliotecas

In [ ]:
# Manipulação de dados
import pandas as pd
import numpy as np

# Vetorização de texto
from sklearn.feature_extraction.text import TfidfVectorizer

# Divisão dos dados e validação cruzada
from sklearn.model_selection import train_test_split, cross_val_score

# Modelos de classificação
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

# Métricas de avaliação
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Visualização
import matplotlib.pyplot as plt
import seaborn as sns

print("Bibliotecas importadas com sucesso!")

### 2. Leitura do Dataset e Análise Exploratória

In [ ]:
# Leitura do CSV (encoding utf-8-sig para compatibilidade com Excel brasileiro)
df = pd.read_csv('dataset_risco.csv', encoding='utf-8-sig')

# Verificar dimensões e tipos
print(f"Total de registros: {len(df)}")
print(f"Colunas: {list(df.columns)}")
print(f"Tipos de dados:\n{df.dtypes}")

# Verificar se há valores nulos
print(f"\nValores nulos por coluna:\n{df.isnull().sum()}")

# Distribuição das classes
print(f"\nDistribuição por classe:")
print(df['situacao'].value_counts())

In [ ]:
# Exibir exemplos de cada classe
print("--- Exemplos de ALTO RISCO ---")
for frase in df[df['situacao'] == 'alto risco']['frase'].head(3):
    print(f"  • {frase}")

print("\n--- Exemplos de BAIXO RISCO ---")
for frase in df[df['situacao'] == 'baixo risco']['frase'].head(3):
    print(f"  • {frase}")

In [ ]:
# Visualização da distribuição das classes
fig, ax = plt.subplots(figsize=(6, 4))

# Cores por classe
cores = {'alto risco': '#e74c3c', 'baixo risco': '#2ecc71'}
contagem = df['situacao'].value_counts()

# Gráfico de barras
contagem.plot(kind='bar', color=[cores[c] for c in contagem.index], ax=ax, edgecolor='black')
ax.set_title('Distribuição das Classes no Dataset', fontsize=14, fontweight='bold')
ax.set_xlabel('Situação')
ax.set_ylabel('Quantidade de Frases')
ax.set_xticklabels(contagem.index, rotation=0)

# Rótulos sobre as barras
for i, v in enumerate(contagem.values):
    ax.text(i, v + 0.2, str(v), ha='center', fontweight='bold', fontsize=12)

plt.tight_layout()
plt.show()

### 3. Pré-processamento do Texto

Convertemos todas as frases para letras minúsculas para padronizar o vocabulário. Sem essa etapa, o modelo trataria "Dor" e "dor" como palavras completamente diferentes, reduzindo a capacidade de generalização.

In [ ]:
# Converter todas as frases para minúsculas e remover espaços extras
df['frase_processada'] = df['frase'].str.lower().str.strip()

# Separar variáveis: X = texto de entrada, y = rótulo de saída
X = df['frase_processada']
y = df['situacao']

# Mostrar o efeito do pré-processamento
print("Pré-processamento concluído!")
print(f"\nExemplo:")
print(f"  Original:   {df['frase'].iloc[0]}")
print(f"  Processada: {df['frase_processada'].iloc[0]}")

### 4. Vetorização com TF-IDF

O **TF-IDF (Term Frequency - Inverse Document Frequency)** transforma cada frase em um vetor numérico. A ideia é:
- **TF**: palavras que aparecem muito em uma frase específica ganham peso maior
- **IDF**: palavras que aparecem em quase todas as frases (ex: "sinto", "estou") ganham peso menor

Usamos `ngram_range=(1, 2)` para capturar não só palavras isoladas (unigramas) como também pares de palavras consecutivas (bigramas). Isso é importante porque "dor peito" carrega mais significado clínico do que "dor" e "peito" separadamente.

In [ ]:
# Configurar o vetorizador TF-IDF
# ngram_range=(1,2): captura unigramas ("dor") e bigramas ("dor peito")
# min_df=1: inclui termos que aparecem em pelo menos 1 documento (necessário com dataset pequeno)
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=1)

# Ajustar o vetorizador ao vocabulário e transformar as frases em vetores
X_tfidf = tfidf.fit_transform(X)

# Verificar as dimensões resultantes
print(f"Dimensão da matriz TF-IDF: {X_tfidf.shape}")
print(f"  → {X_tfidf.shape[0]} frases representadas por {X_tfidf.shape[1]} features (termos)")

# Mostrar alguns termos extraídos
feature_names = tfidf.get_feature_names_out()
print(f"\nExemplos de termos extraídos:")
print(f"  Unigramas: {[t for t in feature_names[:10] if ' ' not in t]}")
print(f"  Bigramas:  {[t for t in feature_names if ' ' in t][:10]}")

### 5. Divisão Treino/Teste

Dividimos os dados em **70% para treino** e **30% para teste**.

Parâmetros importantes:
- `stratify=y`: garante que a proporção de classes (alto/baixo risco) seja mantida nos dois conjuntos
- `random_state=42`: fixa a semente aleatória para que o resultado seja reproduzível

In [ ]:
# Dividir dados em treino (70%) e teste (30%)
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y,
    test_size=0.3,        # 30% para teste
    random_state=42,      # semente para reprodutibilidade
    stratify=y            # manter proporção das classes
)

print(f"Conjunto de TREINO: {X_train.shape[0]} amostras")
print(f"Conjunto de TESTE:  {X_test.shape[0]} amostras")
print(f"\nDistribuição no treino:\n{y_train.value_counts().to_string()}")
print(f"\nDistribuição no teste:\n{y_test.value_counts().to_string()}")

### 6. Treinamento dos Modelos

Treinamos dois modelos para comparação:
- **Logistic Regression**: modelo linear que tende a generalizar bem em datasets textuais pequenos
- **Decision Tree**: modelo baseado em regras — útil para interpretação, mas mais suscetível a overfitting (decorar os dados de treino) em amostras pequenas

In [ ]:
# --- Modelo 1: Logistic Regression (modelo principal) ---
modelo_lr = LogisticRegression(random_state=42, max_iter=1000)
modelo_lr.fit(X_train, y_train)               # treinar com dados de treino
y_pred_lr = modelo_lr.predict(X_test)          # prever nos dados de teste

# --- Modelo 2: Decision Tree (modelo de comparação) ---
modelo_dt = DecisionTreeClassifier(random_state=42, max_depth=5)
modelo_dt.fit(X_train, y_train)
y_pred_dt = modelo_dt.predict(X_test)

# Comparar acurácias
print("Modelos treinados com sucesso!")
print(f"\nAcurácia no teste - Logistic Regression: {accuracy_score(y_test, y_pred_lr):.2%}")
print(f"Acurácia no teste - Decision Tree:        {accuracy_score(y_test, y_pred_dt):.2%}")

### 7. Avaliação do Modelo

#### 7.1 Relatório de Classificação
Além da acurácia, analisamos:
- **Precision**: dos que o modelo classificou como "alto risco", quantos realmente eram?
- **Recall**: dos pacientes que realmente eram "alto risco", quantos o modelo identificou?
- **F1-score**: média harmônica entre precision e recall

In [ ]:
# Definir a ordem das classes explicitamente para evitar inversão
classes = ['alto risco', 'baixo risco']

# Relatório de classificação — Logistic Regression
print("=" * 60)
print("RELATÓRIO DE CLASSIFICAÇÃO — LOGISTIC REGRESSION")
print("=" * 60)
print(f"\nAcurácia geral: {accuracy_score(y_test, y_pred_lr):.2%}")
print(classification_report(y_test, y_pred_lr, labels=classes, target_names=classes))

In [ ]:
# Relatório de classificação — Decision Tree
print("=" * 60)
print("RELATÓRIO DE CLASSIFICAÇÃO — DECISION TREE")
print("=" * 60)
print(f"\nAcurácia geral: {accuracy_score(y_test, y_pred_dt):.2%}")
print(classification_report(y_test, y_pred_dt, labels=classes, target_names=classes))

#### 7.2 Matrizes de Confusão

In [ ]:
# Matrizes de confusão lado a lado
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Logistic Regression
cm_lr = confusion_matrix(y_test, y_pred_lr, labels=classes)
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Reds', ax=axes[0],
            xticklabels=classes, yticklabels=classes)
axes[0].set_title('Logistic Regression', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Predito')
axes[0].set_ylabel('Real')

# Decision Tree
cm_dt = confusion_matrix(y_test, y_pred_dt, labels=classes)
sns.heatmap(cm_dt, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=classes, yticklabels=classes)
axes[1].set_title('Decision Tree', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Predito')
axes[1].set_ylabel('Real')

plt.suptitle('Matrizes de Confusão — Comparação dos Modelos',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

#### 7.3 Validação Cruzada (Cross-Validation)

Com apenas 30 amostras, uma única divisão treino/teste pode não ser confiável — basta 1 erro no teste para a acurácia cair ~11%. A **validação cruzada com 5 folds** repete o processo 5 vezes com divisões diferentes e calcula a média, dando uma estimativa mais robusta do desempenho real.

In [ ]:
# Validação cruzada com 5 folds sobre o dataset completo
cv_lr = cross_val_score(
    LogisticRegression(random_state=42, max_iter=1000),
    X_tfidf, y, cv=5, scoring='accuracy'
)

cv_dt = cross_val_score(
    DecisionTreeClassifier(random_state=42, max_depth=5),
    X_tfidf, y, cv=5, scoring='accuracy'
)

print("VALIDAÇÃO CRUZADA (5 folds)")
print("=" * 50)
print(f"\nLogistic Regression:")
print(f"  Acurácias por fold: {[f'{a:.2%}' for a in cv_lr]}")
print(f"  Média: {cv_lr.mean():.2%} (+/- {cv_lr.std():.2%})")

print(f"\nDecision Tree:")
print(f"  Acurácias por fold: {[f'{a:.2%}' for a in cv_dt]}")
print(f"  Média: {cv_dt.mean():.2%} (+/- {cv_dt.std():.2%})")

### 8. Teste com Frases Inéditas

Testamos o modelo com frases que **não estavam no dataset original** para verificar se ele consegue generalizar para novos relatos. Incluímos propositalmente frases ambíguas para observar o comportamento do classificador.

In [ ]:
# Frases inéditas para teste — incluindo casos ambíguos
frases_novas = [
    "Sinto uma dor intensa no peito e meu braço esquerdo está formigando",
    "Tive uma leve dor de cabeça ontem mas já melhorei",
    "Estou com muita falta de ar e suando frio desde a madrugada",
    "Sinto um desconforto leve nas costas quando fico muito tempo sentado",
    "Meu coração disparou e sinto uma pressão enorme no peito agora",
    "Sinto um leve cansaço e uma dor no peito que aparece de vez em quando"
]

# Aplicar o mesmo pré-processamento usado no treino
frases_processadas = [f.lower().strip() for f in frases_novas]

# Vetorizar usando o mesmo TF-IDF já ajustado (transform, não fit_transform)
X_novas = tfidf.transform(frases_processadas)

# Fazer predições e obter probabilidades
predicoes = modelo_lr.predict(X_novas)
probabilidades = modelo_lr.predict_proba(X_novas)

# Exibir resultados
print("=" * 70)
print("PREDIÇÕES PARA FRASES INÉDITAS")
print("=" * 70)
for i, (frase, pred, prob) in enumerate(zip(frases_novas, predicoes, probabilidades), 1):
    confianca = max(prob) * 100
    indicador = "[ALTO]" if pred == "alto risco" else "[baixo]"
    print(f"\nFrase {i}: \"{frase}\"")
    print(f"   {indicador} Classificação: {pred.upper()} (confiança: {confianca:.1f}%)")

### 9. Análise das Features Mais Relevantes

O modelo de Logistic Regression atribui um **coeficiente (peso)** a cada termo do vocabulário TF-IDF. Coeficientes positivos indicam associação com "baixo risco" e negativos com "alto risco" (ou vice-versa, dependendo da ordenação interna). Visualizar esses pesos ajuda a entender **o que o modelo realmente aprendeu**.

In [ ]:
# Extrair nomes das features e seus coeficientes
feature_names = tfidf.get_feature_names_out()
coeficientes = modelo_lr.coef_[0]

# Identificar os 10 termos com maior peso para cada classe
top_alto = np.argsort(coeficientes)[-10:]    # maiores coeficientes
top_baixo = np.argsort(coeficientes)[:10]    # menores coeficientes

# Visualização lado a lado
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Termos que mais indicam ALTO RISCO
axes[0].barh(range(10), coeficientes[top_alto], color='#e74c3c', edgecolor='black')
axes[0].set_yticks(range(10))
axes[0].set_yticklabels(feature_names[top_alto])
axes[0].set_title('Top 10 — Indicadores de ALTO RISCO', fontweight='bold')
axes[0].set_xlabel('Peso do coeficiente')

# Termos que mais indicam BAIXO RISCO
axes[1].barh(range(10), coeficientes[top_baixo], color='#2ecc71', edgecolor='black')
axes[1].set_yticks(range(10))
axes[1].set_yticklabels(feature_names[top_baixo])
axes[1].set_title('Top 10 — Indicadores de BAIXO RISCO', fontweight='bold')
axes[1].set_xlabel('Peso do coeficiente')

plt.suptitle('Features Mais Relevantes (TF-IDF + Logistic Regression)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 10. Análise de Vieses, Distorções e Limitações

Após a avaliação quantitativa, é fundamental refletir sobre as limitações do modelo e os vieses presentes nos dados e na metodologia.

In [ ]:
# Demonstração prática de viés: testar frases ambíguas
# que misturam sinais de "alto" e "baixo" risco
frases_ambiguas = [
    "sinto uma leve dor no peito que aparece quando faço esforço",
    "tenho dor forte nas costas há uma semana",
    "estou com um leve aperto no tórax desde ontem"
]

X_amb = tfidf.transform(frases_ambiguas)
pred_amb = modelo_lr.predict(X_amb)
prob_amb = modelo_lr.predict_proba(X_amb)

print("TESTE DE VIÉS — FRASES AMBÍGUAS")
print("=" * 60)
for frase, pred, prob in zip(frases_ambiguas, pred_amb, prob_amb):
    confianca = max(prob) * 100
    print(f"\nFrase: \"{frase}\"")
    print(f"  Classificação: {pred} (confiança: {confianca:.1f}%)")
    if confianca < 70:
        print(f"  >>> ATENÇÃO: confiança baixa — modelo inseguro neste caso")

print("\n" + "=" * 60)
print("O modelo tende a usar adjetivos como 'leve' e 'forte' como")
print("atalhos para classificar, em vez de compreender o contexto")
print("clínico real da frase. Isso é um viés de vocabulário.")

#### Limitações identificadas:

1. **Dataset pequeno e simulado:** O modelo foi treinado com apenas 30 frases criadas artificialmente. Em um cenário real, seriam necessários milhares de relatos reais de pacientes para garantir representatividade e robustez estatística.

2. **Vocabulário limitado:** As frases seguem um padrão linguístico restrito. Pacientes reais utilizam gírias regionais, linguagem coloquial variada e descrições menos estruturadas (ex: "meu peito tá apertando demais" em vez de "sinto um aperto no tórax").

3. **Viés de construção (demonstrado acima):** Como as frases foram criadas manualmente, existe um viés implícito na escolha de palavras. Frases de "alto risco" tendem a usar palavras como "forte", "intensa", "súbita", enquanto as de "baixo risco" usam "leve", "pontual", "passou". O modelo pode estar aprendendo esses adjetivos como atalhos em vez de compreender a semântica clínica real.

4. **Ausência de contexto clínico completo:** O modelo não considera idade, histórico médico, exames laboratoriais ou sinais vitais — fatores essenciais para uma triagem real.

5. **Risco de falsos negativos:** Em um sistema clínico real, classificar erroneamente um paciente de "alto risco" como "baixo risco" pode ter consequências graves. Por isso, em aplicações reais, prioriza-se o **recall** da classe "alto risco" sobre a acurácia geral.

#### Conclusão:
Este classificador é um **protótipo educacional** que demonstra a lógica por trás de sistemas reais de triagem. Para uso clínico, seria necessário um dataset robusto com dados reais, validação com profissionais de saúde e auditorias contínuas de viés e equidade.